# Notebook 6 – Population Data Pipeline

## Objective

Develop the population data layer for the Pacific Nutrition Transition Observatory by evaluating, cleaning, validating, and exporting annual population estimates for the sixteen Pacific Island jurisdictions included in this study.

**Primary data source:** United Nations World Population Prospects 2024 (final dataset), following evaluation of the World Bank Open Data API.

**Purpose:** Produce a standardized, quality-assured population dataset for use in downstream data integration, analysis, and visualization within the Pacific Nutrition Transition Observatory.

## 1. Download World Bank population data

In [6]:
import pandas as pd
import requests
from io import BytesIO
import zipfile
import os

os.makedirs("raw_data/WorldBank", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

url = "https://api.worldbank.org/v2/en/indicator/SP.POP.TOTL?downloadformat=csv"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=120
)

print("Status code:", response.status_code)
print("Downloaded MB:", round(len(response.content)/1_000_000, 2))

response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as z:
    print("Files in ZIP:")
    print(z.namelist())

    csv_file = [f for f in z.namelist()
                if f.startswith("API_") and f.endswith(".csv")][0]

    population_raw = pd.read_csv(
        z.open(csv_file),
        skiprows=4
    )

population_raw.to_csv(
    "raw_data/WorldBank/worldbank_population_raw.csv",
    index=False
)

print("Rows downloaded:", len(population_raw))
population_raw.head()

Status code: 200
Downloaded MB: 0.09
Files in ZIP:
['Metadata_Indicator_API_SP.POP.TOTL_DS2_en_csv_v2_430662.csv', 'API_SP.POP.TOTL_DS2_en_csv_v2_430662.csv', 'Metadata_Country_API_SP.POP.TOTL_DS2_en_csv_v2_430662.csv']
Rows downloaded: 266


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,NaN,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,NaN,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,NaN,NaN
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106388440.0,108772632.0,...,440882906.0,452195915.0,463365429.0,474569351.0,485920997.0,497387180.0,509398589.0,521764076.0,NaN,NaN
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,NaN,NaN


## 2. Convert World Bank data from wide to long format

In [7]:
year_columns = [col for col in population_raw.columns if col.isdigit()]

population_long = population_raw.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_columns,
    var_name="Year",
    value_name="Population"
)

population_long = population_long.rename(columns={
    "Country Name": "Country",
    "Country Code": "ISO3"
})

population_long["Year"] = population_long["Year"].astype(int)

print("Rows:", len(population_long))
population_long.head()

Rows: 17556


,Country,ISO3,Indicator Name,Indicator Code,Year,Population
0,Aruba,ABW,"Population, total",SP.POP.TOTL,1960,54922.0
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,1960,130075728.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1960,9035043.0
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,1960,97630925.0
4,Angola,AGO,"Population, total",SP.POP.TOTL,1960,5231654.0


## 3. Filter to Pacific Island jurisdictions

In [8]:
pacific_iso3 = [
    "ASM", "COK", "FJI", "FSM", "PYF",
    "KIR", "MHL", "NRU", "NIU",
    "PLW", "WSM", "SLB", "TKL",
    "TON", "TUV", "VUT"
]

population_pacific = population_long[
    population_long["ISO3"].isin(pacific_iso3)
].copy()

population_pacific = population_pacific[
    ["ISO3", "Country", "Year", "Population"]
]

print("Rows:", len(population_pacific))
population_pacific.head(20)

Rows: 858


,ISO3,Country,Year,Population
11,ASM,American Samoa,1960,20133.0
76,FJI,Fiji,1960,404887.0
79,FSM,"Micronesia, Fed. Sts.",1960,43061.0
124,KIR,Kiribati,1960,47157.0
155,MHL,Marshall Islands,1960,14703.0
179,NRU,Nauru,1960,4607.0
188,PLW,Palau,1960,9328.0
199,PYF,French Polynesia,1960,83671.0
209,SLB,Solomon Islands,1960,139688.0
239,TON,Tonga,1960,66170.0


## 4. Quality assurance

In [9]:
print(population_pacific.info())

print()

print("Countries:")
print(sorted(population_pacific["Country"].unique()))

print()

print("Years:")
print(population_pacific["Year"].min(), "to", population_pacific["Year"].max())

print()

print("Missing values by column:")
print(population_pacific.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 858 entries, 11 to 17550
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ISO3        858 non-null    object 
 1   Country     858 non-null    object 
 2   Year        858 non-null    int64  
 3   Population  845 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 33.5+ KB
None

Countries:
['American Samoa', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Micronesia, Fed. Sts.', 'Nauru', 'Palau', 'Samoa', 'Solomon Islands', 'Tonga', 'Tuvalu', 'Vanuatu']

Years:
1960 to 2025

Missing values by column:
ISO3           0
Country        0
Year           0
Population    13
dtype: int64


## Population source evaluation

The World Bank Open Data API was initially selected because it provides standardized population estimates through a well-documented and easily reproducible interface. Quality assurance of the extracted data revealed that three study jurisdictions (Cook Islands, Niue, and Tokelau) were not represented, resulting in incomplete coverage of the Pacific region.

Because complete geographic coverage is essential for the Pacific Nutrition Transition Observatory, the final population dataset was instead derived from the United Nations World Population Prospects 2024. The UN dataset provides annual population estimates for all sixteen study jurisdictions from 1950 through 2023, making it the authoritative source for the observatory.

## 5. Load United Nations World Population Prospects 2024

The final population dataset is derived from the United Nations World Population Prospects (WPP) 2024. This dataset provides complete coverage for all sixteen Pacific Island jurisdictions included in this project.

In [10]:
import pandas as pd
import os

os.makedirs("raw_data/UN", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

file = "/content/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT.xlsx"

population_raw_un = pd.read_excel(
    file,
    sheet_name="Estimates",
    header=16
)

print("Rows:", len(population_raw_un))
print("Columns:", len(population_raw_un.columns))

population_raw_un.head()

Rows: 21983
Columns: 65


,Index,Variant,"Region, subregion, country or area *",Notes,Location code,ISO3 Alpha-code,ISO2 Alpha-code,SDMX code**,Type,Parent code,...,"Male Mortality before Age 60 (deaths under age 60 per 1,000 male live births)","Female Mortality before Age 60 (deaths under age 60 per 1,000 female live births)","Mortality between Age 15 and 50, both sexes (deaths under age 50 per 1,000 alive at age 15)","Male Mortality between Age 15 and 50 (deaths under age 50 per 1,000 males alive at age 15)","Female Mortality between Age 15 and 50 (deaths under age 50 per 1,000 females alive at age 15)","Mortality between Age 15 and 60, both sexes (deaths under age 60 per 1,000 alive at age 15)","Male Mortality between Age 15 and 60 (deaths under age 60 per 1,000 males alive at age 15)","Female Mortality between Age 15 and 60 (deaths under age 60 per 1,000 females alive at age 15)",Net Number of Migrants (thousands),"Net Migration Rate (per 1,000 population)"
0,1,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,580.5,497.388,238.516,268.734,207.62,375.391,426.221,322.65,0,0
1,2,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,566.566,488.435,229.703,256.236,202.734,365.226,412.76,316.395,0,0
2,3,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,546.444,475.37,217.311,238.56,195.926,350.613,393.364,307.314,0,0
3,4,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,535.811,467.361,211.257,230.961,191.482,342.734,383.875,301.27,0,0
4,5,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,522.058,455.621,203.337,221.377,185.296,332.327,371.737,292.807,0,0


## 6. Filter to Pacific Island jurisdictions

In [11]:
pacific_iso3 = [
    "ASM", "COK", "FJI", "FSM", "PYF",
    "KIR", "MHL", "NRU", "NIU",
    "PLW", "WSM", "SLB", "TKL",
    "TON", "TUV", "VUT"
]

population_pacific = population_raw_un[
    population_raw_un["ISO3 Alpha-code"].isin(pacific_iso3)
].copy()

population_pacific = population_pacific.rename(columns={
    "Region, subregion, country or area *": "Country",
    "ISO3 Alpha-code": "ISO3",
    "Total Population, as of 1 July (thousands)": "Population_Thousands",
    "Population Density, as of 1 July (persons per square km)": "Population_Density",
    "Median Age, as of 1 July (years)": "Median_Age",
    "Population Growth Rate (percentage)": "Population_Growth_Rate"
})

population_pacific = population_pacific[
    [
        "ISO3",
        "Country",
        "Year",
        "Population_Thousands",
        "Population_Density",
        "Median_Age",
        "Population_Growth_Rate"
    ]
]

population_pacific["Population"] = population_pacific["Population_Thousands"] * 1000

population_pacific = population_pacific[
    [
        "ISO3",
        "Country",
        "Year",
        "Population",
        "Population_Thousands",
        "Population_Density",
        "Median_Age",
        "Population_Growth_Rate"
    ]
]

population_pacific["Year"] = population_pacific["Year"].astype(int)

print("Rows:", len(population_pacific))
print("Countries:")
print(sorted(population_pacific["Country"].unique()))
print("Years:", population_pacific["Year"].min(), "to", population_pacific["Year"].max())

population_pacific.head(20)

Rows: 1184
Countries:
['American Samoa', 'Cook Islands', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Micronesia (Fed. States of)', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']
Years: 1950 to 2023


,ISO3,Country,Year,Population,Population_Thousands,Population_Density,Median_Age,Population_Growth_Rate
20281,FJI,Fiji,1950,309514.0,309.514,16.941,16.775,2.031
20282,FJI,Fiji,1951,316123.0,316.123,17.303,16.586,2.194
20283,FJI,Fiji,1952,323365.0,323.365,17.699,16.413,2.334
20284,FJI,Fiji,1953,331215.0,331.215,18.129,16.258,2.462
20285,FJI,Fiji,1954,339745.0,339.745,18.596,16.122,2.621
20286,FJI,Fiji,1955,349070.0,349.07,19.106,16.004,2.792
20287,FJI,Fiji,1956,359210.0,359.21,19.661,15.9,2.933
20288,FJI,Fiji,1957,369913.0,369.913,20.247,15.812,2.94
20289,FJI,Fiji,1958,381072.0,381.072,20.858,15.741,3.003
20290,FJI,Fiji,1959,392760.0,392.76,21.498,15.676,3.038


## 7. Quality assurance (UN dataset)

In [12]:
print(population_pacific.info())

print("\nCountries:")
print(sorted(population_pacific["Country"].unique()))

print("\nYears:")
print(population_pacific["Year"].min(), "to", population_pacific["Year"].max())

print("\nMissing values:")
print(population_pacific.isna().sum())

print("\nDuplicate rows:", population_pacific.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 1184 entries, 20281 to 21908
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   ISO3                    1184 non-null   object
 1   Country                 1184 non-null   object
 2   Year                    1184 non-null   int64 
 3   Population              1184 non-null   object
 4   Population_Thousands    1184 non-null   object
 5   Population_Density      1184 non-null   object
 6   Median_Age              1184 non-null   object
 7   Population_Growth_Rate  1184 non-null   object
dtypes: int64(1), object(7)
memory usage: 115.5+ KB
None

Countries:
['American Samoa', 'Cook Islands', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Micronesia (Fed. States of)', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']

Years:
1950 to 2023

Missing values:
ISO3                      0
Country           

## 8. Standardize data types

In [13]:
numeric_columns = [
    "Population",
    "Population_Thousands",
    "Population_Density",
    "Median_Age",
    "Population_Growth_Rate"
]

for col in numeric_columns:
    population_pacific[col] = pd.to_numeric(
        population_pacific[col],
        errors="coerce"
    )

print(population_pacific.dtypes)

ISO3                       object
Country                    object
Year                        int64
Population                float64
Population_Thousands      float64
Population_Density        float64
Median_Age                float64
Population_Growth_Rate    float64
dtype: object


## 9. Export processed dataset

In [14]:
output_file = "processed_data/population.csv"

population_pacific.to_csv(output_file, index=False)

print("Export complete.")
print(f"File: {output_file}")
print(f"Rows: {len(population_pacific)}")
print(f"Columns: {len(population_pacific.columns)}")

population_pacific.head()

Export complete.
File: processed_data/population.csv
Rows: 1184
Columns: 8


,ISO3,Country,Year,Population,Population_Thousands,Population_Density,Median_Age,Population_Growth_Rate
20281,FJI,Fiji,1950,309514.0,309.514,16.941,16.775,2.031
20282,FJI,Fiji,1951,316123.0,316.123,17.303,16.586,2.194
20283,FJI,Fiji,1952,323365.0,323.365,17.699,16.413,2.334
20284,FJI,Fiji,1953,331215.0,331.215,18.129,16.258,2.462
20285,FJI,Fiji,1954,339745.0,339.745,18.596,16.122,2.621


## Summary

This notebook evaluated two authoritative population data sources for the Pacific Nutrition Transition Observatory.

The World Bank population dataset was initially selected because of its accessible API and standardized structure. Quality assurance revealed that three study jurisdictions (Cook Islands, Niue, and Tokelau) were not represented, making the dataset unsuitable for complete regional analysis.

The final dataset was therefore derived from the United Nations World Population Prospects 2024, which provides complete coverage for all sixteen Pacific jurisdictions from 1950 through 2023. The cleaned dataset was exported for integration with the project's health and food system datasets.